# 11 — Sismo en tiempo real
Recrea con Spark cómo llegaría un sismo a los 6 distritos:
1. **Acelerograma sintético** (método estocástico de Boore) calibrado con los registros de Lima, con validación dejando fuera un sismo.
2. **Escenarios:** el sismo de Callao 2025 (M6.1) y el mismo epicentro con M7.5 y M8.0.
3. **Streaming:** la señal de cada distrito se publica en lotes de 10 s en una tabla Delta (el "tópico"); Spark Structured Streaming la agrega en ventanas de 0.5 s y un STA/LTA detecta la onda P. Se mide cuándo llega la onda P, cuándo empieza la sacudida fuerte y cuánto aviso habría.
4. **Efecto:** la sacudida de cada distrito pasa por la demanda y por el champion del modelo de daño.
5. **Validación** con lo registrado en Callao 2025.

## 1. Setup — imports

In [ ]:
import json
import time
import uuid
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import RegularGridInterpolator
from scipy.signal import butter, cont2discrete, detrend, lfilter, sosfiltfilt, ss2tf
from scipy.signal.windows import tukey

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

## 2. Catálogo, esquema y parámetros

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

In [ ]:
SEED = 700
FILTRO_HZ = (0.1, 25.0)
AMORTIGUAMIENTO = 0.05
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)
G_CM_S2 = 980.665
VS30_GRID = np.arange(150.0, 1001.0, 10.0)
EVENTO_EFECTO = "2025-06-15_callao_M6.1"


def referencia(tipo):
    return pd.DataFrame([json.loads(v) for v in spark.table("gold_referencias").where(F.col("tipo") == tipo).select("valor").toPandas().valor])


REG_LIMA = spark.table("gold_registros").where("ciudad = 'Lima'")
dist = referencia("distritos")

## 3. Funciones: escritura, señal y métricas

In [ ]:
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


def subir_pandas(lotes, tabla, esquema=None, preparar=None, particion=None, mb_lote=64):
    if isinstance(lotes, pd.DataFrame):
        lotes = [lotes]
    modo, n_lotes = "overwrite", 0
    for pdf in lotes:
        if len(pdf) == 0:
            continue
        partes = max(1, int(np.ceil(pdf.memory_usage(deep=True).sum() / 1e6 / mb_lote)))
        for idx in np.array_split(np.arange(len(pdf)), partes):
            sdf = a_spark(pdf.iloc[idx], esquema)
            if preparar is not None:
                sdf = preparar(sdf)
            escribir(sdf, tabla, particion, modo, contar=False)
            modo, n_lotes = "append", n_lotes + 1
    n = spark.table(tabla).count()
    print(f"  {tabla}: {n:,} filas ({n_lotes} lotes)")
    return n


def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append({"tabla": t.tableName, "filas": spark.table(t.tableName).count(), "mb": round((d["sizeInBytes"] or 0) / 1e6, 1)})
    return pd.DataFrame(filas)

In [ ]:
_CACHE_SDOF = {}


def _filtro_sdof(periodo, dt, z):
    if (periodo, dt, z) in _CACHE_SDOF:
        return _CACHE_SDOF[(periodo, dt, z)]
    w = 2 * np.pi / periodo
    A = np.array([[0.0, 1.0], [-w * w, -2 * z * w]])
    B = np.array([[0.0], [-1.0]])
    C = np.array([[1.0, 0.0]])
    D = np.array([[0.0]])
    Ad, Bd, Cd, Dd, _ = cont2discrete((A, B, C, D), dt, method="foh")
    b, a = ss2tf(Ad, Bd, Cd, Dd)
    _CACHE_SDOF[(periodo, dt, z)] = (b[0], a, w)
    return _CACHE_SDOF[(periodo, dt, z)]


def espectro_sa(acc_g, dt, periodos=PERIODOS, z=AMORTIGUAMIENTO):
    sa = np.empty(len(periodos))
    for k, periodo in enumerate(periodos):
        b, a, w = _filtro_sdof(float(periodo), float(dt), z)
        sa[k] = w * w * np.abs(lfilter(b, a, acc_g)).max()
    return sa


def filtrar_cms2(acc_cms2, fs):
    a = detrend(acc_cms2 - acc_cms2.mean(), type="linear")
    a = a * tukey(len(a), 0.05)
    sos = butter(4, FILTRO_HZ, btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, a)


def procesar_componente(acc_cms2, fs):
    dt = 1.0 / fs
    a = filtrar_cms2(acc_cms2, fs)
    vel = cumulative_trapezoid(a, dx=dt, initial=0.0)
    acc_g = a / G_CM_S2
    return acc_g, np.abs(acc_g).max(), np.abs(vel).max(), espectro_sa(acc_g, dt)


def intensidad_arias(acc_g, fs):
    return np.pi / (2 * 9.80665) * np.cumsum((acc_g * 9.80665) ** 2) / fs


def duracion_d595(arias, fs):
    return (np.searchsorted(arias, 0.95 * arias[-1]) - np.searchsorted(arias, 0.05 * arias[-1])) / fs


def variables_onda(x1, x2, fs):
    k = min(len(x1), len(x2))
    pgas, pgvs, sas, arias, d595 = [], [], [], [], []
    for x in (x1[:k], x2[:k]):
        acc_g, pga, pgv, sa = procesar_componente(np.asarray(x, float), fs)
        ia = intensidad_arias(acc_g, fs)
        pgas.append(pga); pgvs.append(pgv); sas.append(sa); arias.append(ia[-1]); d595.append(duracion_d595(ia, fs))
    sa_geo = np.sqrt(sas[0] * sas[1])
    return {"pga_geo_g": float(np.sqrt(pgas[0] * pgas[1])), "pgv_geo_cms": float(np.sqrt(pgvs[0] * pgvs[1])),
            "tp_s": float(PERIODOS[int(sa_geo.argmax())]), "sa_max_geo_g": float(sa_geo.max()), "sa_geo": sa_geo.tolist(),
            "arias_ms": float(np.sqrt(arias[0] * arias[1])), "d595_s": float(np.sqrt(d595[0] * d595[1]))}

In [ ]:
METRICAS = []


def registrar_modelo(nombre_modelo, params, metricas):
    for k, v in metricas.items():
        METRICAS.append({"modelo": nombre_modelo, "metrica": k, "valor": float(v), "parametros": json.dumps(params, default=str)})
    print(f"  {nombre_modelo}: " + " | ".join(f"{k} = {v:,.4f}" for k, v in metricas.items()))


def guardar_metricas(notebook):
    pdf = pd.DataFrame(METRICAS, columns=["modelo", "metrica", "valor", "parametros"])
    pdf["fecha"] = datetime.now().isoformat(timespec="seconds")
    pdf["notebook"] = notebook
    (a_spark(pdf, "modelo string, metrica string, valor double, parametros string, fecha string, notebook string")
     .write.mode("overwrite").option("replaceWhere", f"notebook = '{notebook}'").saveAsTable("ml_metricas"))
    display(pdf)


def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

## 4. Champion de daño y cadena de demanda guardada por 07

In [ ]:
def champion(caso):
    fila = (spark.table("ml_registro_modelos").filter((F.col("caso") == caso) & F.col("promovido"))
            .orderBy(F.col("version").desc()).first())
    if fila is None:
        raise RuntimeError(f"No hay champion de {caso}: ejecute antes su notebook de ML")
    meta = {"version": int(fila["version"]), "algoritmo": fila["algoritmo"], "lote": fila["lote"],
            "cols_num": json.loads(fila["columnas_num"]), "metricas": json.loads(fila["metricas"]), "detalle": json.loads(fila["detalle"])}
    print(f"Champion de {caso}: v{meta['version']} ({meta['algoritmo']}, lote {meta['lote']})")
    return PipelineModel.load(fila["ruta"]), meta


modelo_dano, meta_dano = champion("dano")

In [ ]:
VARS_ONDA = ["PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb"]
RANGO_DANO = meta_dano["detalle"]["rangos"]


def variables_lima(df):
    return (df.withColumn("PGV_cms", F.col("pgv_cms"))
            .withColumn("suelo_cod", F.when(F.col("vs30") < 180, 3.0).when(F.col("vs30") < 500, 2.0).when(F.col("vs30") < 1500, 1.0).otherwise(0.0)))


def fuera_de_rango(columna):
    a, b = RANGO_DANO[columna]
    return ~F.col(columna).between(a, b)


def efecto_por_distrito(df, claves):
    pred = modelo_dano.transform(variables_lima(df)).withColumn("indice_dano", vector_to_array("probability")[1])
    return (pred.groupBy(*claves).agg(
        F.count("*").alias("edificios"),
        F.expr("percentile_approx(PGA_g, 0.5)").alias("PGA_mediana_g"),
        F.expr("percentile_approx(Sa_Tb, 0.5)").alias("Sa_Tb_mediana_g"),
        F.avg("indice_dano").alias("indice_dano_medio"),
        (100 * F.avg((F.col("indice_dano") > 0.5).cast("double"))).alias("pct_indice_mayor_05"),
        *[(100 * F.avg(fuera_de_rango(c).cast("double"))).alias(f"pct_fuera_rango_{c}") for c in VARS_ONDA if c in RANGO_DANO]))

In [ ]:
param = {r["clave"]: json.loads(r["valor"]) for r in spark.table("riesgo_parametros").collect()}
amp = spark.table("riesgo_amplificacion_lima").select("vs30", "indice_periodo", "ln_F", "F").toPandas().sort_values(["vs30", "indice_periodo"])
ln_f = amp.ln_F.to_numpy().reshape(len(VS30_GRID), len(PERIODOS))
F_GRID = amp.F.to_numpy().reshape(len(VS30_GRID), len(PERIODOS))
interp_ln_f = RegularGridInterpolator((VS30_GRID, np.log(PERIODOS)), ln_f)


def ln_amplificacion(T, vs30):
    T, vs30 = np.broadcast_arrays(np.asarray(T, float), np.asarray(vs30, float))
    puntos = np.column_stack([np.clip(vs30.ravel(), VS30_GRID[0], VS30_GRID[-1]),
                              np.log(np.clip(T.ravel(), PERIODOS[0], PERIODOS[-1]))])
    return interp_ln_f(puntos).reshape(T.shape)


PGA_REF_NL = float(param["pga_ref_nl"])
nl = referencia("bssa14").astype(float)
nl = nl[nl.T_s > 0].sort_values("T_s")
F4_NL = np.interp(np.log(PERIODOS), np.log(nl.T_s), nl.f4)
F5_NL = np.interp(np.log(PERIODOS), np.log(nl.T_s), nl.f5)


def ln_F(periodo, vs30, pga_r):
    periodo, vs30, pga_r = np.broadcast_arrays(np.asarray(periodo, float), np.asarray(vs30, float), np.asarray(pga_r, float))
    f4 = np.interp(np.log(periodo), np.log(PERIODOS), F4_NL)
    f5 = np.interp(np.log(periodo), np.log(PERIODOS), F5_NL)
    return ln_amplificacion(periodo, vs30) + f4 * (np.exp(f5 * (np.minimum(vs30, 760.0) - 360.0)) - np.exp(f5 * 400.0)) * (np.log((pga_r + 0.1) / 0.1) - np.log((PGA_REF_NL + 0.1) / 0.1))


tb = np.array([r["Tb"] for r in spark.table("gold_edificios").select("Tb").distinct().collect()], float)
i = np.clip(np.searchsorted(PERIODOS, tb) - 1, 0, len(PERIODOS) - 2)
w = np.clip((np.log(tb) - np.log(PERIODOS[i])) / (np.log(PERIODOS[i + 1]) - np.log(PERIODOS[i])), 0, 1)
tb_tab = a_spark(pd.DataFrame({"Tb": tb, "idx": i + 1, "w": w}), "Tb double, idx int, w double")
amp_arr = a_spark(pd.DataFrame({"vs30_bin": VS30_GRID, "F_arr": [fila.tolist() for fila in F_GRID]}), "vs30_bin double, F_arr array<double>")
sitio_arr = spark.table("riesgo_sitio_h3")
F_SITIO = F.expr("zip_with(F_arr, coalesce(dS_arr, array_repeat(CAST(1.0 AS DOUBLE), size(F_arr))), (a, b) -> a * b)")
arr_sql = lambda v: "array(" + ", ".join(f"{x:.6f}D" for x in v) + ")"
F_NO_LINEAL = F.expr(f"transform(F_arr, (x, i) -> x * exp({arr_sql(F4_NL)}[i] * (exp({arr_sql(F5_NL)}[i] * (least(vs30, 760D) - 360D)) "
                     f"- exp({arr_sql(F5_NL)}[i] * 400D)) * (ln((pga_r_nl + 0.1D) / 0.1D) - {np.log((PGA_REF_NL + 0.1) / 0.1):.6f}D)))")


def interp_log(arreglo):
    return F.exp((1 - F.col("w")) * F.log(F.expr(f"element_at({arreglo}, idx)"))
                 + F.col("w") * F.log(F.expr(f"element_at({arreglo}, idx + 1)")))


def interp_fijo(arreglo, T):
    j = int(np.clip(np.searchsorted(PERIODOS, T) - 1, 0, len(PERIODOS) - 2))
    wj = float(np.clip((np.log(T) - np.log(PERIODOS[j])) / (np.log(PERIODOS[j + 1]) - np.log(PERIODOS[j])), 0, 1))
    return F.exp((1 - wj) * F.log(F.expr(f"element_at({arreglo}, {j + 1})")) + wj * F.log(F.expr(f"element_at({arreglo}, {j + 2})")))


def sa_sitio(T):
    return interp_fijo("sa_roca", T) * interp_fijo("F_arr", T)


ed = (spark.table("gold_edificios")
      .select("id", "ubigeo", "distrito", "provincia", "h3_id", "h3_r8", "pisos", "Tb", "TP", "TL", "S", "Tb_TP", "vs30", "tipo_suelo")
      .where("vs30 IS NOT NULL")
      .withColumn("vs30_bin", F.least(F.lit(float(VS30_GRID.max())),
                                      F.greatest(F.lit(float(VS30_GRID.min())), F.round(F.col("vs30") / 10) * 10))))
print(f"Champion de amplificación usado en 07: v{param['version_amplificacion']} ({param['algoritmo_amplificacion']})")

## 5. Acelerograma sintético calibrado con los registros de Lima

In [ ]:
RHO_G_CM3, BETA_KM_S = 2.8, 3.7
FS_SIM = 100
PERIODOS_CAL = PERIODOS[[int(np.abs(PERIODOS - t).argmin()) for t in (0.05, 0.1, 0.2, 0.3, 0.6, 1.0, 2.0)]]
GRILLA_SINT = [(ds, ka, q0) for ds in (100.0, 300.0, 1000.0, 3000.0) for ka in (0.005, 0.02) for q0 in (300.0, 680.0, 1500.0)]


def espectro_fourier(fr, m, r, ds, ka, q0):
    m0 = 10 ** (1.5 * m + 16.05)
    fc = 4.9e6 * BETA_KM_S * (ds / m0) ** (1 / 3)
    re_km = np.sqrt(r ** 2 + (10 ** (-0.405 + 0.235 * m)) ** 2)
    g = 1 / re_km if re_km <= 70 else (1 / 70 if re_km <= 130 else np.sqrt(130 / re_km) / 70)
    q = np.maximum(q0 * fr ** 0.5, 50.0)
    c = 0.55 * 0.707 * 2.0 / (4 * np.pi * RHO_G_CM3 * BETA_KM_S ** 3) * 1e-20
    return c * m0 * (2 * np.pi * fr) ** 2 / (1 + (fr / fc) ** 2) * g * np.exp(-np.pi * fr * re_km / (q * BETA_KM_S)) * np.exp(-np.pi * ka * fr), fc, re_km


def sintetico_roca(m, r, semilla, ds, ka, q0, correccion=None):
    rng = np.random.default_rng(semilla)
    _, fc, re_km = espectro_fourier(np.array([1.0]), m, r, ds, ka, q0)
    tgm = 1 / fc + 0.05 * re_km
    n = int(2 ** np.ceil(np.log2((2.5 * tgm + 20) * FS_SIM)))
    t = np.arange(n) / FS_SIM
    eps, tn = 0.2, 2 * tgm
    b = -eps * np.log(0.05) / (1 + eps * (np.log(eps) - 1))
    ventana = (np.e / (eps * tn)) ** b * np.maximum(t, 1e-9) ** b * np.exp(-(b / (eps * tn)) * t)
    x = np.fft.rfft(rng.standard_normal(n) * ventana)
    x /= np.sqrt(np.mean(np.abs(x) ** 2))
    fr = np.fft.rfftfreq(n, 1 / FS_SIM)
    fr[0] = fr[1]
    a, _, _ = espectro_fourier(fr, m, r, ds, ka, q0)
    if correccion is not None:
        a = a * np.exp(np.interp(np.log(1 / fr), np.log(PERIODOS_CAL), correccion))
    return np.fft.irfft(a * x * FS_SIM, n)


def ln_sa_sintetico(m, r, semilla, params, correccion=None):
    e = sintetico_roca(m, r, semilla, *params, correccion) / G_CM_S2
    n = sintetico_roca(m, r, semilla + 1, *params, correccion) / G_CM_S2
    return 0.5 * (np.log(espectro_sa(e, 1 / FS_SIM, PERIODOS_CAL)) + np.log(espectro_sa(n, 1 / FS_SIM, PERIODOS_CAL)))


obs_sint = REG_LIMA.where("vs30 IS NOT NULL AND dist_hipo_km IS NOT NULL").select("evento", "vs30", "dist_hipo_km", "sa_geo").toPandas()
obs_sint["m"] = obs_sint.evento.str.extract(r"_M[wl]?(\d+\.\d+)$")[0].astype(float)
obs_sint = pd.concat([g.sample(min(8, len(g)), random_state=SEED) for _, g in obs_sint.groupby("evento")], ignore_index=True)
sa_obs = np.array(obs_sint.sa_geo.tolist(), float)
y_sint = np.array([np.interp(np.log(PERIODOS_CAL), np.log(PERIODOS), np.log(sa_obs[j])) - ln_amplificacion(PERIODOS_CAL, obs_sint.vs30[j]) for j in range(len(obs_sint))])
errores_sint = {g: np.array([ln_sa_sintetico(f.m, f.dist_hipo_km, 7 + f.Index, g) for f in obs_sint.itertuples()]) - y_sint for g in GRILLA_SINT}


def ajustar_sintetico(mascara):
    pesos_ev = 1 / obs_sint[mascara].groupby("evento").evento.transform("count").to_numpy()

    def coste(g):
        c = -np.average(errores_sint[g][mascara], axis=0, weights=pesos_ev)
        return float(np.average(((errores_sint[g][mascara] + c) ** 2).mean(axis=1), weights=pesos_ev))

    mejor_g = min(GRILLA_SINT, key=coste)
    return mejor_g, -np.average(errores_sint[mejor_g][mascara], axis=0, weights=pesos_ev)


filas_val = []
for ev in obs_sint.evento.unique():
    prueba = (obs_sint.evento == ev).to_numpy()
    g_ev, c_ev = ajustar_sintetico(~prueba)
    corregido = np.array([ln_sa_sintetico(f.m, f.dist_hipo_km, 500 + f.Index, g_ev, c_ev) for f in obs_sint[prueba].itertuples()]) - y_sint[prueba]
    filas_val.append({"evento": ev, "delta_sigma": g_ev[0], "kappa": g_ev[1], "q0": g_ev[2], "rmse_ln": rmse(corregido), "sesgo_ln": float(corregido.mean()),
                      "sesgo_ln_t01": float(corregido[:, 1].mean()), "sesgo_ln_t10": float(corregido[:, 5].mean())})
sintetico_val = pd.DataFrame(filas_val)
subir_pandas(sintetico_val, "sismo_sintetico_validacion")
display(sintetico_val.round(3))
PARAM_SINT, C_SINT = ajustar_sintetico(np.ones(len(obs_sint), bool))
registrar_modelo("acelerograma_sintetico", {"delta_sigma": PARAM_SINT[0], "kappa": PARAM_SINT[1], "q0": PARAM_SINT[2], "validacion": "evento excluido", "correccion": dict(zip(PERIODOS_CAL.tolist(), C_SINT.round(3).tolist()))},
                 {"rmse_ln_evento_excluido": float(np.sqrt((sintetico_val.rmse_ln ** 2).mean())), "sesgo_ln_max": float(sintetico_val.sesgo_ln.abs().max())})

## 6. Escenarios

In [ ]:
ev_real = REG_LIMA.where(F.col("evento") == EVENTO_EFECTO).select("ev_lat", "ev_lon", "ev_prof_km").first()
SISMOS_TIEMPO_REAL = [
    {"sismo": "callao_2025_m61", "magnitud": 6.1, "lat": ev_real.ev_lat, "lon": ev_real.ev_lon, "prof_km": ev_real.ev_prof_km, "semilla": SEED, "escenario_real": f"{EVENTO_EFECTO}_registrado"},
    {"sismo": "callao_m75", "magnitud": 7.5, "lat": ev_real.ev_lat, "lon": ev_real.ev_lon, "prof_km": ev_real.ev_prof_km, "semilla": SEED + 1000},
    {"sismo": "callao_m80", "magnitud": 8.0, "lat": ev_real.ev_lat, "lon": ev_real.ev_lon, "prof_km": ev_real.ev_prof_km, "semilla": SEED + 2000},
]
VP_KM_S, VS_KM_S = 7.0, 4.0
RAZON_P_S = 0.15
RUIDO_G = 1e-4
DURACION_MAX_S = 90
LOTE_S = 10
STA_S, LTA_S, UMBRAL_STA_LTA = 0.5, 5.0, 4.0
UMBRAL_SACUDIDA_G = 0.05
BASE_TS = pd.Timestamp("2030-01-01 00:00:00")
VOLUMEN_CHECKPOINT = f"{volumen}/checkpoints"
vs30_distrito = (spark.table("gold_edificios").groupBy("ubigeo").agg(F.expr("percentile_approx(vs30, 0.5)").alias("vs30_mediana")).toPandas())
BASE_DISTRITOS = (dist[dist.en_estudio.astype(bool)][["ubigeo", "distrito", "centroide_lon", "centroide_lat"]].merge(vs30_distrito, on="ubigeo").sort_values("ubigeo").reset_index(drop=True))


def serie_distrito(s, c, j):
    rng = np.random.default_rng(s["semilla"] + 100 * j + 50)
    r = float(np.sqrt(((c.centroide_lon - s["lon"]) * 111.32 * np.cos(np.radians((c.centroide_lat + s["lat"]) / 2))) ** 2
                      + ((c.centroide_lat - s["lat"]) * 110.57) ** 2 + s["prof_km"] ** 2))
    t_p, t_s = r / VP_KM_S, r / VS_KM_S
    roca = [sintetico_roca(s["magnitud"], r, s["semilla"] + 100 * j + i, *PARAM_SINT, C_SINT) for i in (0, 1)]
    n = min(len(x) for x in roca)
    roca = [x[:n] for x in roca]
    pga_r = float(np.sqrt(np.abs(roca[0]).max() * np.abs(roca[1]).max()) / G_CM_S2)
    fr = np.fft.rfftfreq(n, 1 / FS_SIM)
    fr[0] = fr[1]
    transferencia = np.exp(ln_F(1 / fr, c.vs30_mediana, pga_r))
    suelo = [np.fft.irfft(np.fft.rfft(x) * transferencia, n) for x in roca]
    roca_g = [filtrar_cms2(x, FS_SIM) / G_CM_S2 for x in roca]
    suelo_g = [filtrar_cms2(x, FS_SIM) / G_CM_S2 for x in suelo]
    arias = intensidad_arias(suelo_g[0], FS_SIM)
    i_fin = int(min(np.searchsorted(arias, 0.99 * arias[-1]) + 5 * FS_SIM, DURACION_MAX_S * FS_SIM, n))
    i_p, i_s = int(t_p * FS_SIM), int(t_s * FS_SIM)
    columnas = {}
    for nombre, base in (("ew_g", suelo_g[0]), ("ns_g", suelo_g[1]), ("ew_roca_g", roca_g[0]), ("ns_roca_g", roca_g[1])):
        x = rng.normal(0.0, RUIDO_G, i_s + i_fin)
        fase_p = rng.standard_normal(i_s - i_p)
        fase_p = sosfiltfilt(butter(4, (1.0, 25.0), btype="bandpass", fs=FS_SIM, output="sos"), fase_p) * tukey(len(fase_p), 0.2)
        x[i_p:i_s] += fase_p / max(fase_p.std(), 1e-12) * RAZON_P_S * base[:i_fin].std()
        x[i_s:i_s + i_fin] += base[:i_fin]
        columnas[nombre] = x
    return (pd.DataFrame({"sismo": s["sismo"], "ubigeo": c.ubigeo, "distrito": c.distrito, "muestra": np.arange(i_s + i_fin), **columnas}),
            {"sismo": s["sismo"], "ubigeo": c.ubigeo, "distrito": c.distrito, "dist_hipo_km": r, "t_llegada_p_s": t_p, "t_llegada_s_s": t_s, "vs30_mediana": float(c.vs30_mediana)})

## 7. Streaming por distrito

In [ ]:
def guardar_por_sismo(sdf, tabla, sismo):
    if spark.catalog.tableExists(tabla):
        sdf.write.format("delta").mode("overwrite").option("replaceWhere", f"sismo = '{sismo}'").saveAsTable(tabla)
    else:
        sdf.write.format("delta").mode("overwrite").partitionBy("sismo").saveAsTable(tabla)


def correr_sismo(s):
    topico, ventanas = "sismo_stream_acelerograma", "sismo_stream_ventanas"
    checkpoint = f"{VOLUMEN_CHECKPOINT}/{s['sismo']}_{uuid.uuid4().hex[:8]}"
    for t in (topico, ventanas):
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    pares = [serie_distrito(s, c, j) for j, c in enumerate(BASE_DISTRITOS.itertuples())]
    serie = pd.concat([p[0] for p in pares], ignore_index=True)
    llegadas = pd.DataFrame([p[1] for p in pares])
    duracion = (serie.muestra.max() + 1) / FS_SIM

    def consumidor():
        (spark.readStream.table(topico)
         .withWatermark("ts", "1 second")
         .groupBy("sismo", "ubigeo", "distrito", F.window("ts", f"{int(STA_S * 1000)} milliseconds"))
         .agg(F.sum(F.col("ew_g") ** 2 + F.col("ns_g") ** 2).alias("energia"), F.count("*").alias("muestras"), F.max("muestra").alias("ultima_muestra"),
              F.max(F.greatest(F.abs("ew_g"), F.abs("ns_g"))).alias("pico_g"))
         .select("sismo", "ubigeo", "distrito", F.col("window.start").alias("inicio"), "energia", "muestras", "ultima_muestra", "pico_g")
         .writeStream.outputMode("append").option("checkpointLocation", checkpoint)
         .trigger(availableNow=True).toTable(ventanas)
         .awaitTermination())

    def detectar():
        n_lta = int(LTA_S / STA_S)
        previas = Window.partitionBy("ubigeo").orderBy("inicio").rowsBetween(-n_lta, -1)
        return (spark.table(ventanas)
                .withColumn("sta", F.col("energia") / F.col("muestras"))
                .withColumn("lta", F.avg("sta").over(previas)).withColumn("n_lta", F.count("sta").over(previas))
                .withColumn("sta_lta", F.col("sta") / F.col("lta"))
                .where((F.col("n_lta") == n_lta) & (F.col("sta_lta") > UMBRAL_STA_LTA))
                .withColumn("_orden", F.row_number().over(Window.partitionBy("ubigeo").orderBy("inicio"))).where("_orden = 1")
                .select("ubigeo", "ultima_muestra", "sta_lta").toPandas())

    seguimiento, detecciones = [], pd.DataFrame(columns=["ubigeo", "ultima_muestra", "sta_lta"])
    inicio_real = time.perf_counter()
    for k in range(int(np.ceil(duracion / LOTE_S))):
        t0 = time.perf_counter()
        lote = serie[(serie.muestra >= k * LOTE_S * FS_SIM) & (serie.muestra < (k + 1) * LOTE_S * FS_SIM)].copy()
        lote["ts"] = BASE_TS + pd.to_timedelta(lote.muestra / FS_SIM, unit="s")
        a_spark(lote).write.mode("append").saveAsTable(topico)
        consumidor()
        if len(detecciones) < len(BASE_DISTRITOS):
            detecciones = detectar()
        pico = spark.table(ventanas).agg(F.max("pico_g")).first()[0]
        senal = min((k + 1) * LOTE_S, duracion)
        proceso = time.perf_counter() - t0
        seguimiento.append({"sismo": s["sismo"], "lote": k, "senal_s": float(senal), "proceso_s": round(proceso, 1), "reloj_s": round(time.perf_counter() - inicio_real, 1),
                            "distritos_con_p": len(detecciones), "pico_g": None if pico is None else float(pico)})
        print(f"  {s['sismo']} | señal {senal:5.1f} s | proceso {proceso:4.1f} s | P en {len(detecciones)}/{len(BASE_DISTRITOS)} distritos | pico {0.0 if pico is None else pico:.3f} g")
        time.sleep(max(0.0, LOTE_S - (time.perf_counter() - t0)))

    guardar_por_sismo(spark.table(topico), "sismo_acelerogramas", s["sismo"])
    guardar_por_sismo(a_spark(pd.DataFrame(seguimiento), "sismo string, lote long, senal_s double, proceso_s double, reloj_s double, distritos_con_p long, pico_g double"), "sismo_seguimiento", s["sismo"])
    tiempos = (spark.table(topico).withColumn("abs_max", F.greatest(F.abs("ew_g"), F.abs("ns_g"))).groupBy("ubigeo")
               .agg(F.max("abs_max").alias("pico_suelo_g"), F.max_by("muestra", "abs_max").alias("m_pico"),
                    F.min(F.when(F.col("abs_max") >= UMBRAL_SACUDIDA_G, F.col("muestra"))).alias("m_sacudida")).toPandas())
    registro = spark.table(topico).select("ubigeo", "muestra", "ew_g", "ns_g", "ew_roca_g", "ns_roca_g").orderBy("ubigeo", "muestra").toPandas()
    filas_roca, filas_onda = [], []
    for ubigeo, g in registro.groupby("ubigeo"):
        e_r, n_r, e_s, n_s = [g[c].to_numpy(float) for c in ("ew_roca_g", "ns_roca_g", "ew_g", "ns_g")]
        sa_r = np.sqrt(espectro_sa(e_r, 1 / FS_SIM) * espectro_sa(n_r, 1 / FS_SIM))
        pgv_s = float(np.sqrt(np.abs(cumulative_trapezoid(e_s * G_CM_S2, dx=1 / FS_SIM)).max() * np.abs(cumulative_trapezoid(n_s * G_CM_S2, dx=1 / FS_SIM)).max()))
        ia_e, ia_n = intensidad_arias(e_s, FS_SIM), intensidad_arias(n_s, FS_SIM)
        filas_roca.append({"ubigeo": ubigeo, "sa_roca": sa_r.tolist(), "pga_roca_g": float(np.sqrt(np.abs(e_r).max() * np.abs(n_r).max())), "pgv_cms": pgv_s})
        filas_onda.append({"ubigeo": ubigeo, "pgv_suelo_cms": pgv_s, "arias_ms": float(np.sqrt(ia_e[-1] * ia_n[-1])), "d595_s": float(np.sqrt(duracion_d595(ia_e, FS_SIM) * duracion_d595(ia_n, FS_SIM)))})
    roca_s = a_spark(pd.DataFrame(filas_roca), "ubigeo string, sa_roca array<double>, pga_roca_g double, pgv_cms double")
    dem = (ed.where(F.col("ubigeo").isin(BASE_DISTRITOS.ubigeo.tolist()))
           .join(F.broadcast(tb_tab), "Tb").join(F.broadcast(amp_arr), "vs30_bin")
           .join(F.broadcast(sitio_arr), "h3_r8", "left").withColumn("F_arr", F_SITIO)
           .join(F.broadcast(roca_s), "ubigeo").join(spark.table("gold_edificios").select("id", "area_m2"), "id")
           .withColumn("pga_r_nl", F.col("pga_roca_g")).withColumn("F_arr", F_NO_LINEAL)
           .withColumn("Sa_roca_Tb", interp_log("sa_roca")).withColumn("F_Tb", interp_log("F_arr"))
           .withColumn("Sa_Tb", F.col("Sa_roca_Tb") * F.col("F_Tb"))
           .withColumn("PGA_g", F.col("pga_roca_g") * F.expr("element_at(F_arr, 1)"))
           .withColumn("Sa03_g", sa_sitio(0.3)).withColumn("Sa06_g", sa_sitio(0.6)).withColumn("Sa10_g", sa_sitio(1.0)))
    por_distrito = efecto_por_distrito(dem, ["ubigeo", "distrito"]).toPandas()
    detecciones = detecciones.assign(t_deteccion_p_s=(detecciones.ultima_muestra + 1) / FS_SIM)[["ubigeo", "t_deteccion_p_s"]]
    resultado = (llegadas.merge(tiempos, on="ubigeo").merge(pd.DataFrame(filas_onda), on="ubigeo").merge(detecciones, on="ubigeo", how="left").merge(por_distrito.drop(columns="distrito"), on="ubigeo"))
    resultado["t_pico_s"] = resultado.m_pico.astype(float) / FS_SIM
    resultado["t_sacudida_s"] = resultado.m_sacudida.astype(float) / FS_SIM
    resultado["aviso_teorico_s"] = resultado.t_llegada_s_s - resultado.t_deteccion_p_s
    resultado["proceso_medio_por_lote_s"] = float(np.mean([f["proceso_s"] for f in seguimiento]))
    resultado["reloj_hasta_p_s"] = next((f["reloj_s"] for f in seguimiento if f["distritos_con_p"] > 0), np.nan)
    guardar_por_sismo(a_spark(resultado.drop(columns=["m_pico", "m_sacudida"]).assign(magnitud=s["magnitud"])), "sismo_efecto_distritos", s["sismo"])
    for t in (topico, ventanas):
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    return resultado


for s in SISMOS_TIEMPO_REAL:
    display(correr_sismo(s)[["distrito", "dist_hipo_km", "t_llegada_p_s", "t_deteccion_p_s", "t_llegada_s_s", "t_sacudida_s", "t_pico_s", "pico_suelo_g", "PGA_mediana_g", "indice_dano_medio", "pct_indice_mayor_05", "pct_fuera_rango_PGA_g"]].round(3))

## 8. Validación con el sismo de Callao 2025

In [ ]:
s_val = SISMOS_TIEMPO_REAL[0]
sim = spark.table("sismo_efecto_distritos").where(F.col("sismo") == s_val["sismo"]).select("distrito", F.col("PGA_mediana_g").alias("PGA_sintetico"), F.col("Sa_Tb_mediana_g").alias("Sa_Tb_sintetico")).toPandas()
real = (spark.table("riesgo_demanda").where(F.col("escenario") == s_val["escenario_real"]).groupBy("distrito")
        .agg(F.expr("percentile_approx(PGA_g, 0.5)").alias("PGA_registrado"), F.expr("percentile_approx(Sa_Tb, 0.5)").alias("Sa_Tb_registrado")).toPandas())
comparacion = sim.merge(real, on="distrito")
comparacion["razon_PGA"] = comparacion.PGA_sintetico / comparacion.PGA_registrado
comparacion["razon_Sa_Tb"] = comparacion.Sa_Tb_sintetico / comparacion.Sa_Tb_registrado
subir_pandas(comparacion, "sismo_validacion")
display(comparacion.round(3))
registrar_modelo("sismo_tiempo_real", {"lote_s": LOTE_S, "sta_s": STA_S, "lta_s": LTA_S, "umbral": UMBRAL_STA_LTA, "razon_p_s": RAZON_P_S, "validacion": s_val["escenario_real"]},
                 {"rmse_ln_pga_distritos": rmse(np.log(comparacion.razon_PGA)), "rmse_ln_sa_tb_distritos": rmse(np.log(comparacion.razon_Sa_Tb))})

acel = spark.table("sismo_acelerogramas").where(F.col("distrito") == BASE_DISTRITOS.distrito.iloc[0]).select("sismo", "muestra", "ew_g").orderBy("sismo", "muestra").toPandas()
fig, ejes = plt.subplots(len(SISMOS_TIEMPO_REAL), 1, figsize=(11, 2.4 * len(SISMOS_TIEMPO_REAL)), sharex=True)
for eje, s in zip(np.atleast_1d(ejes), SISMOS_TIEMPO_REAL):
    g = acel[acel.sismo == s["sismo"]]
    eje.plot(g.muestra / FS_SIM, g.ew_g, lw=.4)
    eje.set(ylabel="g", title=f"{s['sismo']} (M{s['magnitud']}) en {BASE_DISTRITOS.distrito.iloc[0]}")
eje.set_xlabel("Segundos desde el origen")
plt.tight_layout(); plt.show()
display(spark.table("sismo_efecto_distritos").orderBy("magnitud", F.desc("indice_dano_medio"))
        .select("sismo", "magnitud", "distrito", "t_llegada_p_s", "t_llegada_s_s", "t_sacudida_s", "t_pico_s", "reloj_hasta_p_s", "proceso_medio_por_lote_s", "PGA_mediana_g", "indice_dano_medio", "pct_indice_mayor_05", "pct_fuera_rango_PGA_g").toPandas().round(3))

## 9. Métricas

In [ ]:
guardar_metricas("11_sismo_tiempo_real")